# Model 05: endogamy and assortative mating

This notebook asks how persistent social mating preferences alter genealogical ancestry spread. Communities can occupy the same geographic space and still be reproductively structured.

**Important:** the same-state assortment control is an abstract sensitivity parameter. Deep genealogical ancestry itself is not assumed to be directly observable.

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_DIR = None
if IN_COLAB:
    REPO_DIR = Path('/content/Evolution-Creation')
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git','clone','-q','https://github.com/vafaei-ar/Evolution-Creation.git',str(REPO_DIR)],check=True)
    else:
        subprocess.run(['git','-C',str(REPO_DIR),'fetch','-q','origin','main'],check=True)
        subprocess.run(['git','-C',str(REPO_DIR),'checkout','-q','main'],check=True)
        subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[dev]'],check=True)
else:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / 'src' / 'evolution_creation').exists():
            REPO_DIR = candidate
            break

if REPO_DIR is not None:
    src_path = str(REPO_DIR / 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

print('Environment ready:', REPO_DIR if REPO_DIR is not None else 'using installed Python environment')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from evolution_creation.assortment import (
    deterministic_endogamy_curve,
    make_endogamy_matrix,
    simulate_endogamy_replicates,
)


In [ ]:
n_communities=widgets.IntSlider(value=4,min=2,max=8,description='Communities')
community_size=widgets.IntSlider(value=300,min=50,max=1000,step=50,description='Size/group')
founders=widgets.IntSlider(value=10,min=1,max=50,description='Founders')
endogamy=widgets.FloatSlider(value=0.90,min=0.0,max=1.0,step=0.01,readout_format='.2f',description='Endogamy')
same_state_weight=widgets.FloatLogSlider(value=1.0,base=10,min=-0.7,max=1.0,step=0.05,description='State weight')
generations=widgets.IntSlider(value=40,min=5,max=120,step=5,description='Generations')
replicates=widgets.IntSlider(value=200,min=20,max=1000,step=20,description='Replicates')
seed=widgets.IntText(value=20260920,description='Seed')
display(n_communities,community_size,founders,endogamy,same_state_weight,generations,replicates,seed)

In [ ]:
def run_model(_=None):
    sizes=[community_size.value]*n_communities.value
    matrix=make_endogamy_matrix(sizes,endogamy.value)
    curves=simulate_endogamy_replicates(
        community_sizes=sizes,
        generations=generations.value,
        mate_choice_matrix=matrix,
        founder_community=0,
        founder_count=min(founders.value,community_size.value),
        same_state_weight=same_state_weight.value,
        replicates=replicates.value,
        seed=seed.value,
    )
    deterministic=deterministic_endogamy_curve(
        community_sizes=sizes,
        generations=generations.value,
        mate_choice_matrix=matrix,
        founder_community=0,
        founder_count=min(founders.value,community_size.value),
        same_state_weight=same_state_weight.value,
    )
    x=np.arange(generations.value+1)
    median=np.median(curves,axis=0)
    fig,ax=plt.subplots(figsize=(10,5))
    for c in range(n_communities.value):
        ax.plot(x,median[:,c],label=f'Community {c+1}')
        ax.plot(x,deterministic[:,c],linestyle='--',alpha=.45)
    ax.set(xlabel='Generation',ylabel='Founder-descendant fraction',ylim=(0,1.02))
    ax.legend(ncol=2)
    plt.show()
    final=curves[:,-1,:]
    print(f'Founder lineage survives somewhere: {np.any(final>0,axis=1).mean():.1%}')
    print(f'Every community reached: {np.all(final>0,axis=1).mean():.1%}')
    print(f'Global genealogical fixation: {np.all(final==1,axis=1).mean():.1%}')
    fig,ax=plt.subplots(figsize=(6,5))
    image=ax.imshow(matrix,vmin=0,vmax=1)
    ax.set(xlabel='Mate community',ylabel='Anchor-parent community',title='Mate-choice matrix')
    fig.colorbar(image,ax=ax,label='Probability')
    plt.show()
button=widgets.Button(description='Run simulation',button_style='primary')
button.on_click(run_model)
display(button)
run_model()

## Interpretation

Perfect endogamy is a true reproductive barrier in this model. Near-perfect endogamy is different: ancestry can still cross, but the waiting time for a successful bridge can become long relative to the historical interval being tested.